In [1]:
import os
import time
import numpy as np
import pandas as pd
from sklearn.svm import SVC
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder, LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.metrics import accuracy_score, recall_score, f1_score, precision_score, roc_auc_score
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense, Dropout, Activation
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.utils import to_categorical
import tensorflow as tf
from cleverhans.tf2.attacks.fast_gradient_method import fast_gradient_method

In [ ]:
# ==========================================
# 1. CARREGAMENTO E PRÉ-PROCESSAMENTO 
# ==========================================
print(">>> A carregar datasets...")
df_train = pd.read_csv("data2/BotIoT_training-set.csv")
df_test = pd.read_csv("data2/BotIoT_testing-set.csv")

In [ ]:
# print(">>> A carregar datasets...")
# df_train = pd.read_csv("data/UNSW_NB15_training-set.csv")
# df_test = pd.read_csv("data/UNSW_NB15_testing-set.csv")

In [ ]:


for df in [df_train, df_test]:
    if 'id' in df.columns: df.drop(columns=['id'], inplace=True)

# Binário
y_train_bin = df_train['label'].values
y_test_bin = df_test['label'].values
class_names_bin = ['Normal', 'Attack']

# Multiclasse
df_train['attack_cat'] = df_train['attack_cat'].astype(str).str.strip().str.lower()
df_test['attack_cat'] = df_test['attack_cat'].astype(str).str.strip().str.lower()
le = LabelEncoder()
le.fit(pd.concat([df_train['attack_cat'], df_test['attack_cat']]))
y_train_multi = le.transform(df_train['attack_cat'])
y_test_multi = le.transform(df_test['attack_cat'])
class_names_multi = le.classes_

df_train.drop(columns=['label', 'attack_cat'], inplace=True, errors='ignore')
df_test.drop(columns=['label', 'attack_cat'], inplace=True, errors='ignore')

categorical_cols = df_train.select_dtypes(include=['object']).columns
numerical_cols = df_train.select_dtypes(include=['int64', 'float64']).columns

preprocessor = ColumnTransformer([
    ('num', MinMaxScaler(feature_range=(0,1)), numerical_cols),
    ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_cols)
])

print(">>> A aplicar Scaling e One-Hot Encoding...")
preprocessor.fit(df_train)
X_train = preprocessor.transform(df_train).astype('float32')
X_test = preprocessor.transform(df_test).astype('float32')


>>> A carregar datasets...


C:\Users\Lucas\AppData\Local\Temp\ipykernel_17260\3997558114.py:28: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = df_train.select_dtypes(include=['object']).columns


>>> A aplicar Scaling e One-Hot Encoding...


In [3]:
# ==========================================
# 2. TREINO DA MLP E GERAÇÃO DO ATAQUE FGSM
# ==========================================
def build_and_train_mlp(X, y_cat, num_classes):
    inputs = Input(shape=(X.shape[1],))
    x = Dense(256, activation='relu')(inputs)
    x = Dropout(0.4)(x)
    x = Dense(128, activation='relu')(x)
    x = Dropout(0.4)(x)
    logits = Dense(num_classes, name='logits')(x)
    outputs = Activation('softmax')(logits)
    model = Model(inputs=inputs, outputs=outputs)
    model.compile(loss='categorical_crossentropy', optimizer=Adam(0.001), metrics=['accuracy'])
    model.fit(X, y_cat, batch_size=64, epochs=5, verbose=0)
    return model

epsilon = 0.3

print("\n>>> A gerar FGSM Binário...")
mlp_bin = build_and_train_mlp(X_train, to_categorical(y_train_bin, 2), 2)
logits_bin = Model(inputs=mlp_bin.input, outputs=mlp_bin.get_layer('logits').output)
X_test_adv_bin = fast_gradient_method(logits_bin, tf.convert_to_tensor(X_test), epsilon, np.inf, clip_min=0.0, clip_max=1.0).numpy()

print(">>> A gerar FGSM Multiclasse...")
mlp_multi = build_and_train_mlp(X_train, to_categorical(y_train_multi, len(class_names_multi)), len(class_names_multi))
logits_multi = Model(inputs=mlp_multi.input, outputs=mlp_multi.get_layer('logits').output)
X_test_adv_multi = fast_gradient_method(logits_multi, tf.convert_to_tensor(X_test), epsilon, np.inf, clip_min=0.0, clip_max=1.0).numpy()



>>> A gerar FGSM Binário...
>>> A gerar FGSM Multiclasse...


In [4]:
# ==========================================
# 3. HELPER DE MÉTRICAS (Com AUC e Precisão)
# ==========================================
def calculate_miss_rates(y_true, y_pred, y_prob, context_name, class_names):
    metrics = {}
    
    try:
        normal_idx = next(i for i, name in enumerate(class_names) if 'normal' in str(name).lower())
    except StopIteration:
        normal_idx = 0
        
    is_binary = len(class_names) == 2
    avg_type = 'binary' if is_binary else 'weighted'
    pos_label = 1 if is_binary else None
    
    # Métricas Base
    metrics[f'{context_name}_Acc'] = accuracy_score(y_true, y_pred)
    metrics[f'{context_name}_Precision'] = precision_score(y_true, y_pred, average=avg_type, pos_label=pos_label, zero_division=0)
    metrics[f'{context_name}_Recall'] = recall_score(y_true, y_pred, average=avg_type, pos_label=pos_label, zero_division=0)
    metrics[f'{context_name}_F1'] = f1_score(y_true, y_pred, average=avg_type, pos_label=pos_label, zero_division=0)
    
    if not is_binary:
        metrics[f'{context_name}_F1_Macro'] = f1_score(y_true, y_pred, average='macro', zero_division=0)
        
    # Cálculo do AUC
    try:
        if is_binary:
            prob_positive = y_prob[:, 1] if len(y_prob.shape) > 1 else y_prob
            metrics[f'{context_name}_AUC'] = roc_auc_score(y_true, prob_positive)
        else:
            metrics[f'{context_name}_AUC'] = roc_auc_score(y_true, y_prob, multi_class='ovr')
    except Exception as e:
        metrics[f'{context_name}_AUC'] = 0.0
    
    # Miss Rates
    for idx, name in enumerate(class_names):
        if idx == normal_idx: continue
        mask_true = (y_true == idx)
        if np.sum(mask_true) > 0:
            metrics[f'{context_name}_Miss_{name}'] = np.sum((y_pred == normal_idx) & mask_true) / np.sum(mask_true)
        else:
            metrics[f'{context_name}_Miss_{name}'] = 0.0
            
    return metrics


In [ ]:
# ==========================================
# 4. AVALIAÇÃO DO SUPPORT VECTOR MACHINE (SVM)
# ==========================================
svm_param_grid = {
    'kernel': ['rbf'], # Explicitado no artigo (RBF SVMs)
    'C': [1.0]         # Parâmetro de penalização padrão
}

reports = {'binary': [], 'multiclass': []}

print("\n>>> A iniciar Experimentos: Support Vector Machine (SVM)...")

for mode in ['binary', 'multiclass']:
    print(f"\n  -> 🚀 A correr Pipeline {mode.upper()}...")
    
    if mode == 'binary':
        y_train_curr, y_test_curr = y_train_bin, y_test_bin
        X_adv_curr = X_test_adv_bin
        class_names_curr = class_names_bin
    else:
        y_train_curr, y_test_curr = y_train_multi, y_test_multi
        X_adv_curr = X_test_adv_multi
        class_names_curr = class_names_multi

    for k in svm_param_grid['kernel']:
        for c_val in svm_param_grid['C']:
            
            print(f"     [Kernel={k} | C={c_val}] A treinar o SVM (pode demorar alguns minutos)...")
            
            # probability=True é OBRIGATÓRIO para calcular o AUC
            model = SVC(
                kernel=k,
                C=c_val,
                probability=True, 
                random_state=1
            )
            
            # --- TREINO ---
            t0 = time.time()
            model.fit(X_train, y_train_curr)
            train_time = time.time() - t0
            print(f"       ⏱️ Tempo de treino: {train_time:.2f}s")
            
            # --- INFERÊNCIA LIMPA ---
            t1 = time.time()
            yp_clean = model.predict(X_test)
            yp_clean_prob = model.predict_proba(X_test)
            infer_time_clean = time.time() - t1
            
            # --- INFERÊNCIA ADVERSARIAL ---
            t2 = time.time()
            yp_adv = model.predict(X_adv_curr)
            yp_adv_prob = model.predict_proba(X_adv_curr)
            infer_time_adv = time.time() - t2
            
            # --- MÉTRICAS ---
            m_clean = calculate_miss_rates(y_test_curr, yp_clean, yp_clean_prob, "Clean", class_names_curr)
            m_adv = calculate_miss_rates(y_test_curr, yp_adv, yp_adv_prob, "Adv", class_names_curr)
            acc_drop = m_clean['Clean_Acc'] - m_adv['Adv_Acc']
            
            print(f"       ✅ Queda de Acurácia (Drop): {acc_drop*100:.2f} pp")
            
            row = {
                'Kernel': k, 
                'C_Param': c_val,
                'TrainTime_s': train_time, 
                'InferTime_Clean_s': infer_time_clean, 
                'InferTime_Adv_s': infer_time_adv,    
                'Acc_Drop_pp': acc_drop*100
            }
            row.update(m_clean)
            row.update(m_adv)
            reports[mode].append(row)


>>> A iniciar Experimentos: Support Vector Machine (SVM)...

  -> 🚀 A correr Pipeline BINARY...
     [Kernel=rbf | C=1.0] A treinar o SVM (pode demorar alguns minutos)...
       ⏱️ Tempo de treino: 9.46s
       ✅ Queda de Acurácia (Drop): 0.38 pp

  -> 🚀 A correr Pipeline MULTICLASS...
     [Kernel=rbf | C=1.0] A treinar o SVM (pode demorar alguns minutos)...


In [ ]:
# ==========================================
# 5. EXPORTAÇÃO DOS RELATÓRIOS
# ==========================================
print("\n" + "="*50)
print(">>> GERAÇÃO DE RELATÓRIOS SVM CONCLUÍDA")
print("="*50)

os.makedirs('relatorios BoT-IoT', exist_ok=True)

for mode, data_list in reports.items():
    if not data_list: continue 
    
    df = pd.DataFrame(data_list)
    csv_name = f'relatorios BoT-IoT/svm_{mode}.csv'
    df.to_csv(csv_name, index=False)
    
    print(f"\n📌 RELATÓRIO: SVM - {mode.upper()}")
    print(f"   Salvo em: {csv_name}")
    
    cols_to_show = ['Kernel', 'C_Param', 'TrainTime_s', 'Clean_Acc', 'Adv_Acc', 'Clean_AUC', 'Adv_AUC', 'Acc_Drop_pp']
    
    if mode == 'multiclass' and 'Clean_F1_Macro' in df.columns:
        cols_to_show.insert(5, 'Clean_F1_Macro')
        
    format_dict = {col: '{:.2%}' for col in df.columns if 'Acc' in col or 'F1' in col or 'Recall' in col or 'AUC' in col}
    format_dict['Acc_Drop_pp'] = '{:.2f} pp'
    format_dict['TrainTime_s'] = '{:.2f}s'
    
    display(df[cols_to_show].style.format(format_dict).background_gradient(subset=['Acc_Drop_pp'], cmap='RdYlGn_r'))